In [1]:
import pandas as pd
from glob import glob

runs = glob("./mlruns/**/*.csv", recursive=True)

In [2]:
runs

['./mlruns/336576534199393390/bfcc62521ea74954beac89638536a951/artifacts/commandr-104B-big_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/8b56917191e34e53b63d010c4ce280a8/artifacts/mistral-22B-medium_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/000f0130bcca47edb615e50749baa919/artifacts/commandr-7B-small_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/e00c4d6e195c4e82a4d7f70bcf2fb9d0/artifacts/qwen2.5-72B-big_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/444bedbd45354bc4bea5b312a4250ee7/artifacts/olmo-7B-small_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/868a04798d2448bab5a133913d457b6d/artifacts/mistral-7B-small_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/39d43e70d0eb4ab6aff2238880a57b75/artifacts/mistral-7B-small_subj_checkworthiness_one_shot_llm_pred.csv',
 './mlruns/336576534199393390/d51f65ceef7b4a6eb1904cc4a

In [3]:
cworthy = [r for r in runs if ("checkworthiness" in r) and ("subj" not in r)]
htaxon = [r for r in runs if "hate_taxonomies" in r]

In [4]:
import pandas as pd


replacements = {
    "Check-worthy Factual": "CFS",
    "Non-Factual": "NFS",
    "Unimportant Factual": "UFS",
}
cols = ["premise0", "premise1", "premise2", "conclusion"]

df_orig = pd.read_csv("../data/wsf_annotations_misinformation.csv")

results = {}
for path in cworthy:
    experiment_name = path.split("/")[-1].strip(".csv")
    experiment_name = experiment_name + "_" + path.split("/")[-3]
    results[experiment_name] = {}

    # I've made some updates in the original labels :/
    df = pd.read_csv(path)
    df["check_worthy_premise0"] = df_orig["check_worthy_premise0"]
    df["check_worthy_premise1"] = df_orig["check_worthy_premise1"]
    df["check_worthy_premise2"] = df_orig["check_worthy_premise2"]
    df["check_worthy_conclusion"] = df_orig["check_worthy_conclusion"]

    df = df.rename(
        columns={
            "premise0_llm_pred": "y_pred_premise0_cw",
            "premise1_llm_pred": "y_pred_premise1_cw",
            "premise2_llm_pred": "y_pred_premise2_cw",
            "conclusion_llm_pred": "y_pred_conclusion_cw",
            "check_worthy_premise0": "y_label_premise0_cw",
            "check_worthy_premise1": "y_label_premise1_cw",
            "check_worthy_premise2": "y_label_premise2_cw",
            "check_worthy_conclusion": "y_label_conclusion_cw",
        }
    )

    df["y_pred_premise0_cw"] = df["y_pred_premise0_cw"].replace(replacements)
    df["y_pred_premise1_cw"] = df["y_pred_premise1_cw"].replace(replacements)
    df["y_pred_premise2_cw"] = df["y_pred_premise2_cw"].replace(replacements)
    df["y_pred_conclusion_cw"] = df["y_pred_conclusion_cw"].replace(replacements)

    correct_per_arg = {"args_pred": [], "args_label": [], "args_hate_label": []}
    correct_premises = {"premises_pred": [], "premises_label": [], "premises_hate_label": []}
    for c in cols:
        y_pred = df.loc[~df[c].isna(), f"y_pred_{c}_cw"].replace(replacements)
        y_label = df.loc[~df[c].isna(), f"y_label_{c}_cw"]
        y_hate_label = df.loc[~df[c].isna(), f"{c}_hate"]
        correct_per_arg["args_pred"].extend(y_pred.tolist())
        correct_per_arg["args_label"].extend(y_label.tolist())
        correct_per_arg["args_hate_label"].extend(y_hate_label.tolist())
        if c in ["premise0", "premise1", "premise2"]:
            correct_premises["premises_pred"].extend(y_pred.tolist())
            correct_premises["premises_label"].extend(y_label.tolist())
            correct_premises["premises_hate_label"].extend(y_hate_label.tolist())

        results[experiment_name][c] = (y_label == y_pred).sum() / len(y_label)

        if c in ["conclusion"]:
            results[experiment_name][f"{c}_hate"] = (y_label[y_hate_label == 1] == y_pred[y_hate_label == 1]).sum() / len(y_label[y_hate_label == 1])
            results[experiment_name][f"{c}_nohate"] = (y_label[y_hate_label == 0] == y_pred[y_hate_label == 0]).sum() / len(y_label[y_hate_label == 0])

    correct_per_arg_df = pd.DataFrame(correct_per_arg)
    correct_premises_df = pd.DataFrame(correct_premises)

    results[experiment_name]["all_premises"] = (
        (correct_premises_df["premises_pred"] == correct_premises_df["premises_label"]).sum()
    ) / len(correct_premises_df)
    
    results[experiment_name]["all_args"] = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"]).sum()
    ) / len(correct_per_arg_df)

    nof_CFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "CFS"])
    acc_CFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "CFS")
    ).sum() / nof_CFS
    results[experiment_name]["all_args_CFS"] = acc_CFS

    nof_NFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "NFS"])
    acc_NFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "NFS")
    ).sum() / nof_NFS
    results[experiment_name]["all_args_NFS"] = acc_NFS

    nof_UFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "UFS"])
    acc_UFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "UFS")
    ).sum() / nof_UFS
    results[experiment_name]["all_args_UFS"] = acc_UFS

    premises_hate_df = correct_premises_df[correct_premises_df["premises_hate_label"] == 1]
    premises_nohate_df = correct_premises_df[correct_premises_df["premises_hate_label"] == 0]
    
    results[experiment_name]["all_premises_hate"] = (
        (premises_hate_df["premises_pred"] == premises_hate_df["premises_label"]).sum()
    ) / len(premises_hate_df)
    
    results[experiment_name]["all_premises_nohate"] = (
        (premises_nohate_df["premises_pred"] == premises_nohate_df["premises_label"]).sum()
    ) / len(premises_nohate_df)

    

In [2]:
model_order = {
    "mistral-7B-small_checkworthiness": {
        "zero-shot": ["7cff17b88ad94faa9464b2947940a2d0", "95f14e05f4de44b5bc8de54beeb797e8", "13e8d5a7a90f4df7955bd560ce954cb9"],
        "one-shot": ["995d1484d0464cb2897b8ea1701ce757", "c81f652ad938401cac8dc09b2d4af57e", "40e97314513046ec80d3c564ad524ae1"]
    },
    "llama-8B-small_checkworthiness": {
        "zero-shot": ["4d0f4bdc75d74ee6a1f69beaa5103388", "42d9ae35ca4f4d1db16008bdbfc7a9f7", "daee506eb3694b5788a3ad4dc79be07f"],
        "one-shot": ["bed4838077b24457a96f0761d7beccf7", "2c21dfff23874cae87404fd5abd7e313", "cd696bd8a1eb420e9aaecfa5162195c0"]
    },
    "olmo-7B-small_checkworthiness": {
        "zero-shot": ["3229a4dfda934fda830d0b2effb0298f", "50e85531013f40d7abf041c6ff338351", "a66c91e48b0840ed9182bd75fdebbb67"],
        "one-shot": ["7fa67fe6a9cc4930a52dcc41e028a13a", "38630f13f4ca43928f7839f95b0e5ae5", "72d26a8e05d549269e416358c2477436"],
    },
    "qwen2.5-7B-small_checkworthiness": {
        "zero-shot": ["2686e66957324dcebb68e7e14209461d", "bf33baec21b542a2a17bdced8275fa8b", "de923c3de24f4413a20a6eb3b6ba2a76"],
        "one-shot": ["a158fc90640a48fd87adcf2c32af96cd", "b22d57c18a634050bef8bf13c2df816c", "409bd7acf1b74e07908c54320ef7e0bf"]
    },
    "ommandr-7B-small_checkworthiness": {
        "zero-shot": ["2cb5a96289a54a4a84c4934163cf226b", "7e952d9fd707440ca0c36e127482cc56", "50c78f11aa894e7786df46da3b05a890"],
        "one-shot": ["f300054ab578477e9543926aecdae565", "874ee18a6d284a419800f2743b972707", "672f5a7d65e645b39070b4a4aeeb2a35"],
    },
#    "mixtral-8x7B-small_checkworthiness": {
#        "zero-shot": ["42718aa5c1734282b1d5f86be12abed3"],
#        "one-shot": ["e24034b5591e4d3795247f4cbfc6c475"]
#    },
    "mistral-22B-medium_checkworthiness": {
        "zero-shot": ["d8c0afbd8c6b461496d8e323aea7d846", "d3c2a2ee237648b58324fc4cfc87346a", "dd527efae34c42ab82196cc6f0da9595"],
        "one-shot": ["7251653543d147a490a840ebce03fd21", "0c5129b92b6b4fbd97d6b5ed03d5b0e4", "7b86bb9eae5747ef94c5521e345d0ba5"]
    },
#    "olmo2-32B-medium_checkworthiness": {
#        "zero-shot": ["22493e1f462142fc8b671e105e706036", "31589a761eff46f29f25be22e615ea84"],
#        "one-shot": ["9804734bc67745e9bfab6a3abbda4c66", "74aa782b94554dadbac41b88e2a55e13"]
#    },
#    "mixtral-8x22B-medium_checkworthiness": {
#        "zero-shot": ["ea51bb6edb6c4e79aa215663adb22c13", "255fd733ab3641e19f95c1823ef5923c"],
#        "one-shot": ["6b9fc2f3b96943c4925fef062a7ac144", "99f019dae80742e69a971434369103d0"]
#    },
#    "llama-70B-big_checkworthiness": {
#        "zero-shot": ["ce0707c1e3a0426fbe226774ceac01f7", "c9dcabb0dbde4746b2dfe72c56d24245"],
#        "one-shot": ["b90de0ee45b340c69b609a1809206038", "31051c3dfecc49a88080c77b470b4365"],
#    },
#    "qwen2.5-72B-big_checkworthiness": {
#        "zero-shot": ["e6ca32e4ac724205aeca7c9f7d5ac015", "851646123c0a4150938988be249ba5a3"],
#        "one-shot": ["1f457c7d1aca447fb8c292e0ce7861fc", "014277e19af4488f97128047d4d7b936"]
#    },
#    "ommandr-104B-big_checkworthiness": {
#        "zero-shot": ["99977df7b2044889b1c188ea7941ae5f", "4b38fb694add4ec59378324ecadd4b49"],
#        "one-shot": ["d1f57836592341d7bf1142aa7b767fca", "e89477f022c24d4e855ba2ce01b63f9e"]
#    }
}


# Transform the dictionary
new_model_order = {}

for model_name, data in model_order.items():
    new_model_order[model_name] = {}
    for shot_type, ids in data.items():
        new_urls = []
        for id_ in ids:
            # Find URL containing this id
            matches = [u for u in runs if id_ in u]
            if len(matches) != 1:
                raise ValueError(f"Expected exactly one URL for ID {id_}, found {len(matches)}")
            new_urls.append(matches[0])
        new_model_order[model_name][shot_type] = new_urls

# new_model_order now has URLs instead of IDs
print(new_model_order)


{'mistral-7B-small_checkworthiness': {'zero-shot': ['./mlruns/763756296835180967/7cff17b88ad94faa9464b2947940a2d0/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv', './mlruns/763756296835180967/95f14e05f4de44b5bc8de54beeb797e8/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv', './mlruns/763756296835180967/13e8d5a7a90f4df7955bd560ce954cb9/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv'], 'one-shot': ['./mlruns/529816032717002154/995d1484d0464cb2897b8ea1701ce757/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv', './mlruns/529816032717002154/c81f652ad938401cac8dc09b2d4af57e/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv', './mlruns/529816032717002154/40e97314513046ec80d3c564ad524ae1/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv']}, 'llama-8B-small_checkworthiness': {'zero-shot': ['./mlruns/763756296835180967/4d0f4bdc75d74ee6a1f69beaa5103388/artifacts/llama-8B-small_checkworthiness_zer

In [3]:
from collections import Counter


def majority_vote(row, col):
    values = [row[f"{col}_m0"], row[f"{col}_m1"], row[f"{col}_m2"]]
    counts = Counter(values)
    most_common_value, count = counts.most_common(1)[0]
    if count >= 2:
        return pd.Series({col: most_common_value, f"{col}_agreement_count": count})
    else:
        return pd.Series({col: "All unequal", f"{col}_agreement_count": 1})

runs_w_paths = []
for model_name, data in new_model_order.items():
    for shot_type, urls in data.items():
        df0, df1, df2 = pd.read_csv(urls[0]), pd.read_csv(urls[1]), pd.read_csv(urls[2])
        new_df = df0.copy()
        new_df = new_df.drop(
            columns=["premise0_llm_pred", "premise1_llm_pred", "premise2_llm_pred"]
        )

        new_df["premise0_llm_pred_m0"] = df0["premise0_llm_pred"]
        new_df["premise0_llm_pred_m1"] = df1["premise0_llm_pred"]
        new_df["premise0_llm_pred_m2"] = df2["premise0_llm_pred"]
        new_df[["premise0_llm_pred", "agreement_count"]] = new_df.apply(
            lambda row: majority_vote(row, col="premise0_llm_pred"), axis=1
        )

        new_df["premise1_llm_pred_m0"] = df0["premise1_llm_pred"]
        new_df["premise1_llm_pred_m1"] = df1["premise1_llm_pred"]
        new_df["premise1_llm_pred_m2"] = df2["premise1_llm_pred"]
        new_df[["premise1_llm_pred", "agreement_count"]] = new_df.apply(
            lambda row: majority_vote(row, col="premise1_llm_pred"), axis=1
        )

        new_df["premise2_llm_pred_m0"] = df0["premise2_llm_pred"]
        new_df["premise2_llm_pred_m1"] = df1["premise2_llm_pred"]
        new_df["premise2_llm_pred_m2"] = df2["premise2_llm_pred"]
        new_df[["premise2_llm_pred", "agreement_count"]] = new_df.apply(
            lambda row: majority_vote(row, col="premise2_llm_pred"), axis=1
        )

        new_df["conclusion_llm_pred_m0"] = df0["conclusion_llm_pred"]
        new_df["conclusion_llm_pred_m1"] = df1["conclusion_llm_pred"]
        new_df["conclusion_llm_pred_m2"] = df2["conclusion_llm_pred"]
        new_df[["conclusion_llm_pred", "agreement_count"]] = new_df.apply(
            lambda row: majority_vote(row, col="conclusion_llm_pred"), axis=1
        )
        runs_w_paths.append((new_df, urls))

In [6]:
import pandas as pd


replacements = {
    "Check-worthy Factual": "CFS",
    "Non-Factual": "NFS",
    "Unimportant Factual": "UFS",
}
cols = ["premise0", "premise1", "premise2", "conclusion"]

df_orig = pd.read_csv("../data/wsf_annotations_misinformation.csv")

results = {}
for df, path in runs_w_paths:
    experiment_name = path[0].split("/")[-1].strip(".csv")
    experiment_name = experiment_name # + "_" + path.split("/")[-3]
    results[experiment_name] = {}

    df["check_worthy_premise0"] = df_orig["check_worthy_premise0"]
    df["check_worthy_premise1"] = df_orig["check_worthy_premise1"]
    df["check_worthy_premise2"] = df_orig["check_worthy_premise2"]
    df["check_worthy_conclusion"] = df_orig["check_worthy_conclusion"]

    df = df.rename(
        columns={
            "premise0_llm_pred": "y_pred_premise0_cw",
            "premise1_llm_pred": "y_pred_premise1_cw",
            "premise2_llm_pred": "y_pred_premise2_cw",
            "conclusion_llm_pred": "y_pred_conclusion_cw",
            "check_worthy_premise0": "y_label_premise0_cw",
            "check_worthy_premise1": "y_label_premise1_cw",
            "check_worthy_premise2": "y_label_premise2_cw",
            "check_worthy_conclusion": "y_label_conclusion_cw",
        }
    )

    df["y_pred_premise0_cw"] = df["y_pred_premise0_cw"].replace(replacements)
    df["y_pred_premise1_cw"] = df["y_pred_premise1_cw"].replace(replacements)
    df["y_pred_premise2_cw"] = df["y_pred_premise2_cw"].replace(replacements)
    df["y_pred_conclusion_cw"] = df["y_pred_conclusion_cw"].replace(replacements)

    correct_per_arg = {"args_pred": [], "args_label": [], "args_hate_label": []}
    correct_premises = {"premises_pred": [], "premises_label": [], "premises_hate_label": []}
    for c in cols:
        y_pred = df.loc[~df[c].isna(), f"y_pred_{c}_cw"].replace(replacements)
        y_label = df.loc[~df[c].isna(), f"y_label_{c}_cw"]
        y_hate_label = df.loc[~df[c].isna(), f"{c}_hate"]
        correct_per_arg["args_pred"].extend(y_pred.tolist())
        correct_per_arg["args_label"].extend(y_label.tolist())
        correct_per_arg["args_hate_label"].extend(y_hate_label.tolist())
        if c in ["premise0", "premise1", "premise2"]:
            correct_premises["premises_pred"].extend(y_pred.tolist())
            correct_premises["premises_label"].extend(y_label.tolist())
            correct_premises["premises_hate_label"].extend(y_hate_label.tolist())

        results[experiment_name][c] = (y_label == y_pred).sum() / len(y_label)

        if c in ["conclusion"]:
            results[experiment_name][f"{c}_hate"] = (y_label[y_hate_label == 1] == y_pred[y_hate_label == 1]).sum() / len(y_label[y_hate_label == 1])
            results[experiment_name][f"{c}_nohate"] = (y_label[y_hate_label == 0] == y_pred[y_hate_label == 0]).sum() / len(y_label[y_hate_label == 0])

    correct_per_arg_df = pd.DataFrame(correct_per_arg)
    correct_premises_df = pd.DataFrame(correct_premises)

    results[experiment_name]["all_premises"] = (
        (correct_premises_df["premises_pred"] == correct_premises_df["premises_label"]).sum()
    ) / len(correct_premises_df)
    
    results[experiment_name]["all_args"] = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"]).sum()
    ) / len(correct_per_arg_df)

    nof_CFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "CFS"])
    acc_CFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "CFS")
    ).sum() / nof_CFS
    results[experiment_name]["all_args_CFS"] = acc_CFS

    nof_NFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "NFS"])
    acc_NFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "NFS")
    ).sum() / nof_NFS
    results[experiment_name]["all_args_NFS"] = acc_NFS

    nof_UFS = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "UFS"])
    acc_UFS = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "UFS")
    ).sum() / nof_UFS
    results[experiment_name]["all_args_UFS"] = acc_UFS

    premises_hate_df = correct_premises_df[correct_premises_df["premises_hate_label"] == 1]
    premises_nohate_df = correct_premises_df[correct_premises_df["premises_hate_label"] == 0]
    
    results[experiment_name]["all_premises_hate"] = (
        (premises_hate_df["premises_pred"] == premises_hate_df["premises_label"]).sum()
    ) / len(premises_hate_df)
    
    results[experiment_name]["all_premises_nohate"] = (
        (premises_nohate_df["premises_pred"] == premises_nohate_df["premises_label"]).sum()
    ) / len(premises_nohate_df)

    

In [7]:
df_results = pd.DataFrame(results).transpose()

In [9]:
df_results[["premise0", "premise1", "premise2", "all_premises", "conclusion", "all_args", "all_args_CFS", "all_args_NFS", "all_args_UFS", "all_premises_hate", "all_premises_nohate", "conclusion_hate", "conclusion_nohate"]]

,premise0,premise1,premise2,all_premises,conclusion,all_args,all_args_CFS,all_args_NFS,all_args_UFS,all_premises_hate,all_premises_nohate,conclusion_hate,conclusion_nohate
mistral-7B-small_checkworthiness_zero_shot_llm_pred,0.261062,0.278146,0.275862,0.268473,0.484581,0.345972,0.095890,0.993939,0.196078,0.197970,0.334928,0.449704,0.586207
mistral-7B-small_checkworthiness_one_shot_llm_pred,0.389381,0.350993,0.241379,0.364532,0.484581,0.407583,0.219178,0.806061,0.441176,0.269036,0.454545,0.461538,0.551724
llama-8B-small_checkworthiness_zero_shot_llm_pred,0.194690,0.218543,0.206897,0.204433,0.449339,0.292259,0.027397,0.987879,0.117647,0.192893,0.215311,0.426036,0.517241
llama-8B-small_checkworthiness_one_shot_llm_pred,0.225664,0.238411,0.310345,0.236453,0.480176,0.323855,0.082192,0.963636,0.156863,0.213198,0.258373,0.455621,0.551724
olmo-7B-small_checkworthiness_zero_shot_llm_pred,0.371681,0.331126,0.310345,0.352217,0.453744,0.388626,0.213699,0.854545,0.264706,0.269036,0.430622,0.437870,0.500000
olmo-7B-small_checkworthiness_one_shot_llm_pred,0.252212,0.211921,0.275862,0.238916,0.471366,0.322275,0.090411,0.963636,0.117647,0.208122,0.267943,0.449704,0.534483
qwen2.5-7B-small_checkworthiness_zero_shot_llm_pred,0.296460,0.284768,0.241379,0.288177,0.497797,0.363349,0.104110,0.963636,0.323529,0.243655,0.330144,0.473373,0.568966
qwen2.5-7B-small_checkworthiness_one_shot_llm_pred,0.243363,0.225166,0.206897,0.233990,0.449339,0.311216,0.054795,0.993939,0.127451,0.213198,0.253589,0.431953,0.500000
ommandr-7B-small_checkworthiness_zero_shot_llm_pred,0.424779,0.350993,0.379310,0.394089,0.572687,0.458136,0.419178,0.800000,0.049020,0.411168,0.377990,0.573964,0.568966
ommandr-7B-small_checkworthiness_one_shot_llm_pred,0.371681,0.350993,0.310345,0.359606,0.537445,0.423381,0.339726,0.854545,0.029412,0.375635,0.344498,0.532544,0.551724


In [5]:
path

['./mlruns/763756296835180967/7cff17b88ad94faa9464b2947940a2d0/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv',
 './mlruns/763756296835180967/95f14e05f4de44b5bc8de54beeb797e8/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv',
 './mlruns/763756296835180967/13e8d5a7a90f4df7955bd560ce954cb9/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv']

In [44]:
new_model_order

{'mistral-7B-small_checkworthiness': {'zero-shot': ['./mlruns/763756296835180967/7cff17b88ad94faa9464b2947940a2d0/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv',
   './mlruns/763756296835180967/95f14e05f4de44b5bc8de54beeb797e8/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv',
   './mlruns/763756296835180967/13e8d5a7a90f4df7955bd560ce954cb9/artifacts/mistral-7B-small_checkworthiness_zero_shot_llm_pred.csv'],
  'one-shot': ['./mlruns/529816032717002154/995d1484d0464cb2897b8ea1701ce757/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv',
   './mlruns/529816032717002154/c81f652ad938401cac8dc09b2d4af57e/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv',
   './mlruns/529816032717002154/40e97314513046ec80d3c564ad524ae1/artifacts/mistral-7B-small_checkworthiness_one_shot_llm_pred.csv']},
 'llama-8B-small_checkworthiness': {'zero-shot': ['./mlruns/763756296835180967/4d0f4bdc75d74ee6a1f69beaa5103388/artifacts/llama-8B-small_chec

In [ ]:
relevant_runs_dfs = []
for path in cworthy:
    experiment_name = path.split("/")[-1].strip(".csv")
    experiment_name = experiment_name + "_" + path.split("/")[-3]
    df = pd.read_csv(path)
    relevant_runs_dfs.append(df)

In [5]:
model_order = {
    "mistral-7B-small_checkworthiness": {
        "zero-shot": ["7cff17b88ad94faa9464b2947940a2d0", "95f14e05f4de44b5bc8de54beeb797e8", "13e8d5a7a90f4df7955bd560ce954cb9"],
        "one-shot": ["995d1484d0464cb2897b8ea1701ce757", "c81f652ad938401cac8dc09b2d4af57e", "40e97314513046ec80d3c564ad524ae1"]
    },
    "llama-8B-small_checkworthiness": {
        "zero-shot": ["4d0f4bdc75d74ee6a1f69beaa5103388", "42d9ae35ca4f4d1db16008bdbfc7a9f7", "daee506eb3694b5788a3ad4dc79be07f"],
        "one-shot": ["bed4838077b24457a96f0761d7beccf7", "2c21dfff23874cae87404fd5abd7e313", "cd696bd8a1eb420e9aaecfa5162195c0"]
    },
    "olmo-7B-small_checkworthiness": {
        "zero-shot": ["3229a4dfda934fda830d0b2effb0298f", "50e85531013f40d7abf041c6ff338351", "a66c91e48b0840ed9182bd75fdebbb67"],
        "one-shot": ["7fa67fe6a9cc4930a52dcc41e028a13a", "38630f13f4ca43928f7839f95b0e5ae5", "72d26a8e05d549269e416358c2477436"],
    },
    "qwen2.5-7B-small_checkworthiness": {
        "zero-shot": ["2686e66957324dcebb68e7e14209461d", "bf33baec21b542a2a17bdced8275fa8b", "de923c3de24f4413a20a6eb3b6ba2a76"],
        "one-shot": ["a158fc90640a48fd87adcf2c32af96cd", "b22d57c18a634050bef8bf13c2df816c", "409bd7acf1b74e07908c54320ef7e0bf"]
    },
    "ommandr-7B-small_checkworthiness": {
        "zero-shot": ["2cb5a96289a54a4a84c4934163cf226b", "7e952d9fd707440ca0c36e127482cc56", "50c78f11aa894e7786df46da3b05a890"],
        "one-shot": ["f300054ab578477e9543926aecdae565", "874ee18a6d284a419800f2743b972707", "672f5a7d65e645b39070b4a4aeeb2a35"],
    },
    "mixtral-8x7B-small_checkworthiness": {
        "zero-shot": ["42718aa5c1734282b1d5f86be12abed3"],
        "one-shot": ["e24034b5591e4d3795247f4cbfc6c475"]
    },
    "mistral-22B-medium_checkworthiness": {
        "zero-shot": ["d8c0afbd8c6b461496d8e323aea7d846", "d3c2a2ee237648b58324fc4cfc87346a", "dd527efae34c42ab82196cc6f0da9595"],
        "one-shot": ["7251653543d147a490a840ebce03fd21", "0c5129b92b6b4fbd97d6b5ed03d5b0e4", "7b86bb9eae5747ef94c5521e345d0ba5"]
    },
    "olmo2-32B-medium_checkworthiness": {
        "zero-shot": ["22493e1f462142fc8b671e105e706036", "31589a761eff46f29f25be22e615ea84"],
        "one-shot": ["9804734bc67745e9bfab6a3abbda4c66", "74aa782b94554dadbac41b88e2a55e13"]
    },
    "mixtral-8x22B-medium_checkworthiness": {
        "zero-shot": ["ea51bb6edb6c4e79aa215663adb22c13", "255fd733ab3641e19f95c1823ef5923c"],
        "one-shot": ["6b9fc2f3b96943c4925fef062a7ac144", "99f019dae80742e69a971434369103d0"]
    },
    "llama-70B-big_checkworthiness": {
        "zero-shot": ["ce0707c1e3a0426fbe226774ceac01f7", "c9dcabb0dbde4746b2dfe72c56d24245"],
        "one-shot": ["b90de0ee45b340c69b609a1809206038", "31051c3dfecc49a88080c77b470b4365"],
    },
    "qwen2.5-72B-big_checkworthiness": {
        "zero-shot": ["e6ca32e4ac724205aeca7c9f7d5ac015", "851646123c0a4150938988be249ba5a3"],
        "one-shot": ["1f457c7d1aca447fb8c292e0ce7861fc", "014277e19af4488f97128047d4d7b936"]
    },
    "ommandr-104B-big_checkworthiness": {
        "zero-shot": ["99977df7b2044889b1c188ea7941ae5f", "4b38fb694add4ec59378324ecadd4b49"],
        "one-shot": ["d1f57836592341d7bf1142aa7b767fca", "e89477f022c24d4e855ba2ce01b63f9e"]
    }
}

# --- Build the full desired order list ---
desired_order = []

for model, shots in model_order.items():
    for shot_type in ["zero-shot", "one-shot"]:  # always keep this order
        if shot_type in shots:
            for exp_id in shots[shot_type]:
                # Construct the pattern that matches your dataframe index
                # Example: "mistral-7B-small_checkworthiness_zero_shot_llm_pred_<id>"
                name = f"{model}_{shot_type.replace('-', '_')}_llm_pred_{exp_id}"
                desired_order.append(name)

# --- Now reorder dataframe safely ---
# Keep only those rows that exist in df (in case some are missing)

df_results = pd.DataFrame(results).transpose()

valid_order = [i for i in desired_order if i in df_results.index]

df_sorted = df_results.loc[valid_order]

df_sorted[["premise0", "premise1", "premise2", "all_premises", "conclusion", "all_args", "all_args_CFS", "all_args_NFS", "all_args_UFS", "all_premises_hate", "all_premises_nohate", "conclusion_hate", "conclusion_nohate"]].to_csv("normal_out.csv")

In [6]:
df_sorted[["premise0", "premise1", "premise2", "all_premises", "conclusion", "all_args", "all_args_CFS", "all_args_NFS", "all_args_UFS", "all_premises_hate", "all_premises_nohate", "conclusion_hate", "conclusion_nohate"]]

,premise0,premise1,premise2,all_premises,conclusion,all_args,all_args_CFS,all_args_NFS,all_args_UFS,all_premises_hate,all_premises_nohate,conclusion_hate,conclusion_nohate
mistral-7B-small_checkworthiness_zero_shot_llm_pred_7cff17b88ad94faa9464b2947940a2d0,0.261062,0.278146,0.275862,0.268473,0.484581,0.345972,0.095890,0.993939,0.196078,0.197970,0.334928,0.449704,0.586207
mistral-7B-small_checkworthiness_zero_shot_llm_pred_95f14e05f4de44b5bc8de54beeb797e8,0.261062,0.278146,0.275862,0.268473,0.484581,0.345972,0.095890,0.993939,0.196078,0.197970,0.334928,0.449704,0.586207
mistral-7B-small_checkworthiness_zero_shot_llm_pred_13e8d5a7a90f4df7955bd560ce954cb9,0.261062,0.271523,0.275862,0.266010,0.484581,0.344392,0.095890,0.993939,0.186275,0.197970,0.330144,0.449704,0.586207
mistral-7B-small_checkworthiness_one_shot_llm_pred_995d1484d0464cb2897b8ea1701ce757,0.389381,0.350993,0.275862,0.366995,0.475771,0.406003,0.210959,0.812121,0.450980,0.269036,0.459330,0.449704,0.551724
mistral-7B-small_checkworthiness_one_shot_llm_pred_c81f652ad938401cac8dc09b2d4af57e,0.380531,0.350993,0.241379,0.359606,0.484581,0.404423,0.213699,0.806061,0.441176,0.258883,0.454545,0.461538,0.551724
mistral-7B-small_checkworthiness_one_shot_llm_pred_40e97314513046ec80d3c564ad524ae1,0.393805,0.350993,0.241379,0.366995,0.484581,0.409163,0.219178,0.812121,0.441176,0.269036,0.459330,0.461538,0.551724
llama-8B-small_checkworthiness_zero_shot_llm_pred_4d0f4bdc75d74ee6a1f69beaa5103388,0.194690,0.218543,0.206897,0.204433,0.449339,0.292259,0.027397,0.987879,0.117647,0.192893,0.215311,0.426036,0.517241
llama-8B-small_checkworthiness_zero_shot_llm_pred_42d9ae35ca4f4d1db16008bdbfc7a9f7,0.194690,0.218543,0.241379,0.206897,0.458150,0.296998,0.032877,0.981818,0.137255,0.192893,0.220096,0.431953,0.534483
llama-8B-small_checkworthiness_zero_shot_llm_pred_daee506eb3694b5788a3ad4dc79be07f,0.194690,0.218543,0.206897,0.204433,0.449339,0.292259,0.027397,0.987879,0.117647,0.192893,0.215311,0.426036,0.517241
llama-8B-small_checkworthiness_one_shot_llm_pred_bed4838077b24457a96f0761d7beccf7,0.225664,0.231788,0.310345,0.233990,0.480176,0.322275,0.079452,0.963636,0.156863,0.213198,0.253589,0.455621,0.551724


In [ ]:
df_results

In [37]:
df_results

,premise0,premise1,premise2,conclusion,conclusion_hate,conclusion_nohate,all_premises,all_args,all_args_CFS,all_args_NFS,all_args_UFS,all_premises_hate,all_premises_nohate
mixtral-8x7B-small_checkworthiness_zero_shot_llm_pred_42718aa5c1734282b1d5f86be12abed3,0.623894,0.576159,0.344828,0.563877,0.579882,0.517241,0.586207,0.578199,0.696970,0.558282,0.217822,0.593909,0.578947
olmo2-32B-medium_checkworthiness_zero_shot_llm_pred_22493e1f462142fc8b671e105e706036,0.615044,0.543046,0.310345,0.625551,0.615385,0.655172,0.566502,0.587678,0.622590,0.736196,0.257426,0.517766,0.612440
olmo-7B-small_checkworthiness_zero_shot_llm_pred_3229a4dfda934fda830d0b2effb0298f,0.358407,0.284768,0.172414,0.466960,0.437870,0.551724,0.317734,0.371248,0.187328,0.877301,0.237624,0.253807,0.377990
mistral-22B-medium_checkworthiness_zero_shot_llm_pred_a8a9c0451b49433ea5cd920fb68e3649,0.261062,0.211921,0.137931,0.493392,0.479290,0.534483,0.233990,0.327014,0.115702,0.987730,0.039604,0.213198,0.253589
mistral-22B-medium_checkworthiness_zero_shot_llm_pred_d8c0afbd8c6b461496d8e323aea7d846,0.265487,0.211921,0.172414,0.493392,0.479290,0.534483,0.238916,0.330174,0.121212,0.987730,0.039604,0.218274,0.258373
olmo-7B-small_checkworthiness_zero_shot_llm_pred_50e85531013f40d7abf041c6ff338351,0.371681,0.337748,0.206897,0.449339,0.437870,0.482759,0.347291,0.383886,0.212121,0.852761,0.267327,0.269036,0.421053
mistral-7B-small_checkworthiness_zero_shot_llm_pred_7cff17b88ad94faa9464b2947940a2d0,0.261062,0.278146,0.172414,0.480176,0.449704,0.568966,0.261084,0.339652,0.093664,0.993865,0.188119,0.192893,0.325359
llama-8B-small_checkworthiness_zero_shot_llm_pred_4d0f4bdc75d74ee6a1f69beaa5103388,0.194690,0.218543,0.137931,0.449339,0.426036,0.517241,0.199507,0.289100,0.027548,0.987730,0.118812,0.187817,0.210526
ommandr-7B-small_checkworthiness_zero_shot_llm_pred_2cb5a96289a54a4a84c4934163cf226b,0.424779,0.344371,0.241379,0.563877,0.562130,0.568966,0.381773,0.447077,0.407713,0.809816,0.029703,0.406091,0.358852
llama-70B-big_checkworthiness_zero_shot_llm_pred_ce0707c1e3a0426fbe226774ceac01f7,0.486726,0.377483,0.241379,0.546256,0.520710,0.620690,0.428571,0.470774,0.336088,0.987730,0.148515,0.350254,0.502392


In [30]:
df_sorted[["premise0", "premise1", "premise2", "all_premises", "conclusion", "all_args", "all_args_CFS", "all_args_NFS", "all_args_UFS", "all_premises_hate", "all_premises_nohate", "conclusion_hate", "conclusion_nohate"]]

,premise0,premise1,premise2,all_premises,conclusion,all_args,all_args_CFS,all_args_NFS,all_args_UFS,all_premises_hate,all_premises_nohate,conclusion_hate,conclusion_nohate


In [184]:

# Example: df is your dataframe with those indices
# df = pd.DataFrame(...)

# Custom order for models
model_order = [
    "mistral-7B",
    "llama-8B",
    "olmo-7B",      # (you wrote olmo2-7b, but in your list it's `olmo-7B`)
    "qwen2.5-7B",
    "ommandr-7B",
    "mixtral-8x7B",
    "mistral-22B",
    "olmo2-32B",
    "mixtral-8x22B",
    "llama-70B",
    "qwen2.5-72B",
    "ommandr-104B"
]

# Custom order for shot type
shot_order = ["zero_shot", "one_shot"]

# Build a ranking dict for sorting
order_dict = {
    f"{model}_{shot}": i
    for model_idx, model in enumerate(model_order)
    for shot_idx, shot in enumerate(shot_order)
    for i in [model_idx * len(shot_order) + shot_idx]
}

# Function to extract the relevant key
def extract_key(idx):
    for model in model_order:
        if idx.startswith(model):
            shot = "zero_shot" if "zero_shot" in idx else "one_shot"
            return f"{model}_{shot}"
    return None

# Sort index based on our ranking
df_res = pd.DataFrame(results).transpose()
df_res_sorted = df_res.loc[sorted(df_res.index, key=lambda x: order_dict.get(extract_key(x), 1e9))]


,premise0,premise1,premise2,all_premises,conclusion,all_args,all_args_CFS,all_args_NFS,all_args_UFS,all_premises_hate,all_premises_nohate,conclusion_hate,conclusion_nohate
mistral-7B-small_checkworthiness_zero_shot_llm_pred__7cff17b88ad94faa9464b2947940a2d0,0.261062,0.278146,0.172414,0.261084,0.480176,0.339652,0.093664,0.993865,0.188119,0.192893,0.325359,0.449704,0.568966
mistral-7B-small_checkworthiness_zero_shot_llm_pred__13e8d5a7a90f4df7955bd560ce954cb9,0.261062,0.271523,0.172414,0.258621,0.480176,0.338073,0.093664,0.993865,0.178218,0.192893,0.320574,0.449704,0.568966
mistral-7B-small_checkworthiness_one_shot_llm_pred__995d1484d0464cb2897b8ea1701ce757,0.389381,0.350993,0.206897,0.362069,0.471366,0.401264,0.209366,0.815951,0.445545,0.263959,0.454545,0.449704,0.534483
mistral-7B-small_checkworthiness_one_shot_llm_pred__40e97314513046ec80d3c564ad524ae1,0.393805,0.350993,0.172414,0.362069,0.480176,0.404423,0.217631,0.815951,0.435644,0.263959,0.454545,0.461538,0.534483
llama-8B-small_checkworthiness_zero_shot_llm_pred__42d9ae35ca4f4d1db16008bdbfc7a9f7,0.194690,0.218543,0.172414,0.201970,0.453744,0.292259,0.033058,0.981595,0.128713,0.187817,0.215311,0.431953,0.517241
llama-8B-small_checkworthiness_zero_shot_llm_pred__daee506eb3694b5788a3ad4dc79be07f,0.194690,0.218543,0.137931,0.199507,0.449339,0.289100,0.027548,0.987730,0.118812,0.187817,0.210526,0.426036,0.517241
llama-8B-small_checkworthiness_one_shot_llm_pred__bed4838077b24457a96f0761d7beccf7,0.225664,0.231788,0.206897,0.226601,0.480176,0.317536,0.077135,0.963190,0.158416,0.208122,0.244019,0.455621,0.551724
llama-8B-small_checkworthiness_one_shot_llm_pred__2c21dfff23874cae87404fd5abd7e313,0.225664,0.238411,0.206897,0.229064,0.480176,0.319115,0.079890,0.963190,0.158416,0.208122,0.248804,0.455621,0.551724
olmo-7B-small_checkworthiness_zero_shot_llm_pred__3229a4dfda934fda830d0b2effb0298f,0.358407,0.284768,0.172414,0.317734,0.466960,0.371248,0.187328,0.877301,0.237624,0.253807,0.377990,0.437870,0.551724
olmo-7B-small_checkworthiness_zero_shot_llm_pred__a66c91e48b0840ed9182bd75fdebbb67,0.371681,0.331126,0.206897,0.344828,0.449339,0.382306,0.212121,0.852761,0.257426,0.263959,0.421053,0.437870,0.482759


In [185]:
df_res_sorted[["premise0", "premise1", "premise2", "all_premises", "conclusion", "all_args", "all_args_CFS", "all_args_NFS", "all_args_UFS", "all_premises_hate", "all_premises_nohate", "conclusion_hate", "conclusion_nohate"]].to_csv("normal_out.csv")

In [24]:
import pandas as pd

cols = ["premise0", "premise1", "premise2", "conclusion"]

results = {}
for path in htaxon:
    experiment_name = path.split("/")[-1].strip(".csv")
    results[experiment_name] = {}

    df = pd.read_csv(path)

    df = df.rename(
        columns={
            "premise0_llm_pred": "y_pred_premise0_cw",
            "premise1_llm_pred": "y_pred_premise1_cw",
            "premise2_llm_pred": "y_pred_premise2_cw",
            "conclusion_llm_pred": "y_pred_conclusion_cw",
            "hate_taxon_premise0": "y_label_premise0_cw",
            "hate_taxon_premise1": "y_label_premise1_cw",
            "hate_taxon_premise2": "y_label_premise2_cw",
            "hate_taxon_conclusion": "y_label_conclusion_cw",
        }
    )

    correct_per_arg = {"args_pred": [], "args_label": []}
    for c in cols:
        y_pred = df.loc[~df[c].isna(), f"y_pred_{c}_cw"]
        y_label = df.loc[~df[c].isna(), f"y_label_{c}_cw"]
        correct_per_arg["args_pred"].extend(y_pred.tolist())
        correct_per_arg["args_label"].extend(y_label.tolist())

        results[experiment_name][c] = (y_label == y_pred).sum() / len(y_label)

    correct_per_arg_df = pd.DataFrame(correct_per_arg)
    results[experiment_name]["all_args"] = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"]).sum()
    ) / len(correct_per_arg_df)

    nof_wg = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "White Grievance"])
    acc_wg = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "White Grievance")
    ).sum() / nof_wg
    results[experiment_name]["all_args_white_grievance"] = acc_wg

    nof_iv = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "Incitement to Violence"])
    acc_iv = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "Incitement to Violence")
    ).sum() / nof_iv
    results[experiment_name]["all_args_incitement_violence"] = acc_iv

    nof_il = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "Inferiority Language"])
    acc_il = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "Inferiority Language")
    ).sum() / nof_il
    results[experiment_name]["all_args_inferiority_language"] = acc_il

    nof_iro = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "Irony"])
    acc_iro = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "Irony")
    ).sum() / nof_iro
    results[experiment_name]["all_args_inferiority_language"] = acc_iro

    nof_stereotype = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "Stereotypes and Misinformation"])
    acc_stereotype = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "Stereotypes and Misinformation")
    ).sum() / nof_stereotype
    results[experiment_name]["all_args_inferiority_language"] = acc_stereotype

    nof_threat = len(correct_per_arg_df[correct_per_arg_df["args_label"] == "Threatening and Intimidation"])
    acc_threat = (
        (correct_per_arg_df["args_pred"] == correct_per_arg_df["args_label"])
        & (correct_per_arg_df["args_label"] == "Threatening and Intimidation")
    ).sum() / nof_threat
    results[experiment_name]["all_args_inferiority_language"] = acc_threat

KeyError: 'y_label_premise0_cw'

["White Grievance", "Incitement to Violence", "Inferiority Language", "Irony", "Stereotypes and Misinformation", "Threatening and Intimidation"]

In [25]:
df

,file_id,idx,text_ed,IS,premise0,check_worthy_premise0,truth-o-meter_premise0,comments_premise0,premise1,check_worthy_premise1,...,premise0_hate,premise1_hate,premise2_hate,conclusion_hate,arg_comps_ok,target_ok,y_pred_premise0_cw,y_pred_premise1_cw,y_pred_premise2_cw,y_pred_conclusion_cw
0,30679934_1,0,the kind of perversion that leads to homosexua...,gay men are perverts.,The kind of perversion that leads to homosexua...,CFS,pants-fire,association fallacy?,Gay men may turn out to be pedophiles.,CFS,...,1.0,1.0,0.0,1.0,1.0,1.0,Threatening and Intimidation,Threatening and Intimidation,Irony,Threatening and Intimidation
1,13470210_2,1,The Jews say they are The Chosen Ones The Blac...,jewish people are inferior to other groups.,The Jews believe they are The Chosen Ones.,CFS,false?,stereotype? generalizes and potentially misrep...,The Blacks claim to be the original Hebrews an...,CFS,...,0.0,0.0,0.0,0.0,1.0,1.0,Inferiority Language,Stereotypes and Misinformation,Irony,Threatening and Intimidation
2,12834493_1,2,She may or may not be a Jew but she 's certain...,jewish folks are stupid.,she seems to think the Blacks wo n't kill her ...,NFS,NaN,speculation about others' intentions,NaN,NaN,...,1.0,0.0,0.0,1.0,1.0,1.0,Threatening and Intimidation,Irony,Irony,Inferiority Language
3,13251831_1,3,I always thought it would be cool if somebody ...,Indians and Pakis should be killed.,It would be cool if someone secretly detonated...,NFS,NaN,"expresses a wish or hypothetical desire, not a...",NaN,NaN,...,1.0,0.0,0.0,1.0,0.0,1.0,Inferiority Language,Irony,Irony,Incitement to Violence
4,14039975_1,4,Was walking through tooting broadway in SW Lon...,black folks are inferior.,I was the only white person in the area throug...,CFS,false?,"makes a personal, observational claim about a ...",Nigs and pakis were walking down the street sp...,UFS,...,1.0,1.0,0.0,1.0,1.0,1.0,Irony,Threatening and Intimidation,Irony,Inferiority Language
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,31723707_3,224,There is only about 10 niggas in our school an...,black folks and hispanics are inferior.,there are only 10 niggas in our school and 300...,CFS,NaN,NaN,it is the senior year.,CFS,...,1.0,0.0,0.0,1.0,NaN,NaN,Threatening and Intimidation,Threatening and Intimidation,Threatening and Intimidation,Threatening and Intimidation
223,31760414_1,225,Looking at your guys situation makes me realis...,black folks are invading us.,my high school is 100% white.,CFS,NaN,are there schools 100% white?,there are more and more niggas moving into the...,CFS,...,0.0,1.0,0.0,1.0,NaN,NaN,Inferiority Language,Inferiority Language,Irony,White Grievance
224,13480579_1,226,"just saw a hershey 's commercial on tv , ca n'...",jews are inferior.,i just saw a hershey's commercial on tv.,UFS,NaN,NaN,i can't find it on youtube.,UFS,...,0.0,0.0,0.0,0.0,NaN,NaN,Irony,Inferiority Language,Inferiority Language,Threatening and Intimidation
225,13843470_1,227,"These are the latest statistics , Ethnic group...",people of color are inferior.,"irish 87.4%, other white 7.5%, asian 1.3%, bla...",CFS,NaN,NaN,even if that seven percent was all poles.,NFS,...,0.0,0.0,0.0,0.0,NaN,NaN,Stereotypes and Misinformation,Stereotypes and Misinformation,Irony,Stereotypes and Misinformation
